# Embeddings
https://platform.openai.com/docs/models#embeddings

임베딩(Embeddings)은 텍스트를 수치적으로 표현한 값으로, 두 텍스트 간의 연관성을 측정하는 데 사용된다.

임베딩은 검색, 군집화(clustering), 추천 시스템, 이상 탐지, 분류와 같은 작업에 유용하다.

**모델 및 출력 차원**

| 모델 이름                     | 설명                                                              | 출력 차원 |
|-------------------------------|-------------------------------------------------------------------|-----------|
| **text-embedding-3-large**   | 영어 및 비영어 작업 모두에서 가장 강력한 성능을 가진 모델           | 3,072     |
| **text-embedding-3-small**   | 2세대 ada 임베딩 모델보다 성능이 향상된 모델                        | 1,536     |
| **text-embedding-ada-002**   | 1세대 모델 16개를 대체하는 가장 강력한 2세대 임베딩 모델             | 1,536     |


## MTEB Leaderboard
**Massive Text Embedding Benchmark (MTEB) Leaderboard**

https://huggingface.co/spaces/mteb/leaderboard

**MTEB Leaderboard**는 Hugging Face에서 제공하는 벤치마크 리더보드 페이지로, 다양한 언어 모델(Language Model)과 임베딩 모델(Embedding Model)의 성능을 객관적으로 비교·평가하는 공간이다.

**MTEB Leaderboard에서 순위 산정 방식**

**MTEB Leaderboard**의 순위는 다양한 자연어 처리 태스크(분류, 클러스터링, 검색, 문장 유사도 등)에서 모델이 얻은 점수들의 평균을 기반으로 산정된다. 즉, 여러 벤치마크 데이터셋에서 모델의 성능을 측정하고, 이를 종합하여 평균 점수를 계산한 뒤, 이 평균 점수가 높은 순서대로 모델이 정렬된다.

**주요 평가 방식**

- **평가 태스크 종류**
  - 분류(Classification): F1 점수
  - 클러스터링(Clustering): V-measure
  - 쌍 분류(Pair Classification): Average Precision
  - 재정렬(Reranking): MRR@k, MAP
  - 검색(Retrieval): nDCG@k
  - 의미 유사도(STS): Spearman correlation
  - 요약(Summarization): Spearman correlation  
  각 태스크별로 대표적인 평가 지표가 다르며, 모델은 여러 태스크에서 평가를 받는다[2].

- **평균 점수 산정**
  - 각 태스크별로 모델이 얻은 점수를 모두 합산한 뒤, 태스크 수로 나누어 평균 점수를 구한다.
  - 이 평균 점수가 리더보드의 기본 순위 기준이 된다.

- **부분 평가 가능**
  - 모든 태스크를 수행하지 않아도 특정 태스크만 평가받아 부분 리더보드에 오를 수 있다. 예를 들어, 클러스터링 태스크만 평가받아 해당 부분 순위에 표시될 수 있다.

In [2]:
from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv()  # .env파일 읽어 환경변수 등록
client = OpenAI()  # Openai api 응답 객체

In [3]:
response = client.embeddings.create(
    model='text-embedding-3-small',
    input=["Hello World!", "안녕하세요?"]
)

response

CreateEmbeddingResponse(data=[Embedding(embedding=[-0.0030193328857421875, -0.0567626953125, 0.02947998046875, 0.042999267578125, -0.04083251953125, -0.025177001953125, -0.01282501220703125, 0.03515625, -0.03155517578125, -0.01108551025390625, -0.0158843994140625, -0.031097412109375, -0.020294189453125, -0.024688720703125, 0.029632568359375, 0.035888671875, -0.038177490234375, 0.017822265625, 0.01137542724609375, 0.040771484375, 0.04754638671875, 0.0025005340576171875, -0.006389617919921875, -0.01392364501953125, 0.03485107421875, -0.01221466064453125, -0.04437255859375, 0.0185089111328125, 0.0237274169921875, -0.0433349609375, 0.044952392578125, -0.036224365234375, -0.010223388671875, 0.005977630615234375, 0.00621795654296875, 0.0006060600280761719, -0.001708984375, 0.00647735595703125, -0.0014619827270507812, -0.02398681640625, 0.0121612548828125, -0.0277862548828125, 0.01025390625, 0.040313720703125, -0.05206298828125, 0.0291748046875, -0.05596923828125, 0.01230621337890625, 0.03167

In [ ]:
len(response.data[0].embedding)  # 첫 번째 문장의 임베딩 벡터 길이

1536

In [ ]:
emb_vec0 = response.data[0].embedding  # 첫 번째 문장의 임베딩 벡터
emb_vec1 = response.data[1].embedding

print(len(emb_vec0))  # 첫 번째 문장의 임베딩 벡터 길이
print(len(emb_vec1))

1536
1536


In [7]:
def text_to_embedding(texts, model='text-embedding-3-small'):
    texts = [text.replace("\n", ' ') for text in texts]
    response = client.embeddings.create(model=model, input=texts)

    return [data.embedding for data in response.data]

vecs = text_to_embedding(["Hello World!", "안녕하세요?"])

print(len(vecs[0]))
print(len(vecs[1]))

1536
1536


### 음식리뷰 임베딩 처리

https://www.kaggle.com/datasets/snap/amazon-fine-food-reviews   

corpus를 임베딩 벡터로 변환하고, 이후에는 이 임베딩 벡터로 VectorDB에 저장해 놓고, 유사도를 통한 검색이 가능하다.   

In [9]:
import pandas as pd

review_df = pd.read_csv('fine_food_reviews_1k.csv', index_col=0)
print(review_df.info())
review_df.head()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   Time       1000 non-null   int64
 1   ProductId  1000 non-null   str  
 2   UserId     1000 non-null   str  
 3   Score      1000 non-null   int64
 4   Summary    1000 non-null   str  
 5   Text       1000 non-null   str  
dtypes: int64(2), str(4)
memory usage: 450.7 KB
None


,Time,ProductId,UserId,Score,Summary,Text
0,1351123200,B003XPF9BO,A3R7JR3FMEBXQB,5,where does one start...and stop... with a tre...,Wanted to save some to bring to my Chicago fam...
1,1351123200,B003JK537S,A3JBPC3WFUT5ZP,1,Arrived in pieces,"Not pleased at all. When I opened the box, mos..."
2,1351123200,B000JMBE7M,AQX1N6A51QOKG,4,"It isn't blanc mange, but isn't bad . . .",I'm not sure that custard is really custard wi...
3,1351123200,B004AHGBX4,A2UY46X0OSNVUQ,3,These also have SALT and it's not sea salt.,I like the fact that you can see what you're g...
4,1351123200,B001BORBHO,A1AFOYZ9HSM2CZ,5,Happy with the product,My dog was suffering with itchy skin. He had ...


In [10]:
review_df = review_df[['Summary', 'Text']]

review_df['Content'] = 'Title: ' + review_df['Summary'].str.strip() + '; Content: ' + review_df['Text'].str.strip()
review_df

,Summary,Text,Content
0,where does one start...and stop... with a tre...,Wanted to save some to bring to my Chicago fam...,Title: where does one start...and stop... wit...
1,Arrived in pieces,"Not pleased at all. When I opened the box, mos...",Title: Arrived in pieces; Content: Not pleased...
2,"It isn't blanc mange, but isn't bad . . .",I'm not sure that custard is really custard wi...,"Title: It isn't blanc mange, but isn't bad . ...."
3,These also have SALT and it's not sea salt.,I like the fact that you can see what you're g...,Title: These also have SALT and it's not sea s...
4,Happy with the product,My dog was suffering with itchy skin. He had ...,Title: Happy with the product; Content: My dog...
...,...,...,...
995,Delicious!,I have ordered these raisins multiple times. ...,Title: Delicious!; Content: I have ordered the...
996,Good Training Treat,My dog will come in from outside when I am tra...,Title: Good Training Treat; Content: My dog wi...
997,Jamica Me Crazy Coffee,Wolfgang Puck's Jamaica Me Crazy is that wonde...,Title: Jamica Me Crazy Coffee; Content: Wolfga...
998,Party Peanuts,Great product for the price. Mix with the Asia...,Title: Party Peanuts; Content: Great product f...


In [11]:
# Content를 리스트로 변환 후 임베딩 생성 -> embedding 컬럼 저장
review_df['embedding'] = text_to_embedding(review_df['Content'].tolist())
review_df.head()

,Summary,Text,Content,embedding
0,where does one start...and stop... with a tre...,Wanted to save some to bring to my Chicago fam...,Title: where does one start...and stop... wit...,"[0.036651611328125, -0.023193359375, -0.030471..."
1,Arrived in pieces,"Not pleased at all. When I opened the box, mos...",Title: Arrived in pieces; Content: Not pleased...,"[0.01140594482421875, 0.0343017578125, -0.0411..."
2,"It isn't blanc mange, but isn't bad . . .",I'm not sure that custard is really custard wi...,"Title: It isn't blanc mange, but isn't bad . ....","[0.0032253265380859375, 0.01265716552734375, -..."
3,These also have SALT and it's not sea salt.,I like the fact that you can see what you're g...,Title: These also have SALT and it's not sea s...,"[-0.0028896331787109375, 0.01462554931640625, ..."
4,Happy with the product,My dog was suffering with itchy skin. He had ...,Title: Happy with the product; Content: My dog...,"[0.01204681396484375, -0.0560302734375, 0.0167..."


In [ ]:
embed_df = review_df['embedding'].to_frame('embedding')  # Series(embedding) -> DataFrame 변환
embed_df.index = review_df['Content']  # 인덱스는 Content 텍스트
embed_df

,embedding
Content,
Title: where does one start...and stop... with a treat like this; Content: Wanted to save some to bring to my Chicago family but my North Carolina family ate all 4 boxes before I could pack. These are excellent...could serve to anyone,"[0.036651611328125, -0.023193359375, -0.030471..."
"Title: Arrived in pieces; Content: Not pleased at all. When I opened the box, most of the rings were broken in pieces. A total waste of money.","[0.01140594482421875, 0.0343017578125, -0.0411..."
"Title: It isn't blanc mange, but isn't bad . . .; Content: I'm not sure that custard is really custard without eggs. But this comes close. I got it for use in a ""Vegan pancake"" recipe. We were having houseguests who were Vegan and I wanted to make some special breakfasts while they were here. One of the cooking/recipe sites had a recipe using this and there were lots of great reviews. I tried the recipe and it turned out like wallpaper paste -- yuck!<br />However, the so-called custard isn't so bad. I think it's probably just cornstarch and annatto (yellow coloring with a slight flavor). It's fun playing with it. You could dress it up with fruit. Seems to come out on the thin side when you make it as directed, so I use less milk because I like my custards to set firm. As a custard sauce it's fine. I would say it tastes something between a pudding and a custard.<br /><br />If you want a really good egg-free ""custard"" get an original recipe for ""blanc mange."" It takes a lot longer to make, but it's certainly worth the difference.","[0.0032253265380859375, 0.01265716552734375, -..."
"Title: These also have SALT and it's not sea salt.; Content: I like the fact that you can see what you're getting and that there are no bones or dark meat. There are 7 nice big chunks in every jar.<br /><br />These taste like tuna in a can but, because they're preserved in glass, you don't have to worry about either aluminum or BPA; BUT ... they are not just tuna and spring water.<br /><br />There is salt in there, too, and it's not healthy sea salt, it's toxic table salt.<br /><br />I am trying to contact Tonnino to confirm that. I might be wrong because the label states that the ingredients are ""tuna fish"" but the sticker on the top clarifies that it is the smaller (healthier) yellowfin, so the ""salt"" listed in the ingredients might be sea salt but, if it was, why don't they say so?<br /><br />Without confirmation, I will continue to look for a salt-free olive-oil free tuna preserved in glass.<br /><br />If you know of one, please contact me!","[-0.0028896331787109375, 0.01462554931640625, ..."
Title: Happy with the product; Content: My dog was suffering with itchy skin. He had been eating Natural Choice brand (cheaper) since he was a puppy. I was nervous to change foods. The vet suggested to change foods sand see if the skin issues cleared up. Wellness brand did the job. My dog seems to love the food and the skin issues cleared up within a few weeks.,"[0.01204681396484375, -0.0560302734375, 0.0167..."
...,...
Title: Delicious!; Content: I have ordered these raisins multiple times. They are always great and arrive timely. I can't go back to store bought chocolate covered raisins now! Love this product.,"[0.016448974609375, -0.039154052734375, -0.026..."
Title: Good Training Treat; Content: My dog will come in from outside when I am training her and look at the cupboard waiting for her treat. When I use the clicker training method she comes because she knows she has something special.,"[-0.023040771484375, -0.0139007568359375, 0.00..."
Title: Jamica Me Crazy Coffee; Content: Wolfgang Puck's Jamaica Me Crazy is that wonderful blend of island flavors in a coffee. Have loved it from the first time tasting. Great product.,"[-0.02960205078125, -0.045745849609375, -0.026..."


In [ ]:
# 유사도 검색 (Vector Search 기반 검색)
from sklearn.metrics.pairwise import cosine_similarity

# 쿼리와 가장 유사한 리뷰를 벡터 유사도로 Top-n개 검색하는 함수
def review_vector_search(query, embed_df=embed_df, top_n=5):
    query_emb = text_to_embedding([query])  # 사용자 쿼리를 임베딩 벡터로 변환

    df = embed_df.copy()
    df['cos_sim'] = df['embedding'].apply(lambda emb: cosine_similarity([emb], query_emb)[0, 0])

    df = df.sort_values('cos_sim', ascending=False).head(top_n)  # 유사도 높은 순 상위 n개
    df = df.reset_index()  # 인덱스 : Content -> 0부터 재생성 (Content는 컬럼으로 변환)
    df = df[['Content', 'cos_sim']]

    return df

pd.set_option('display.max_colwidth', None)  # Content가 잘리는 것 방지 (최대 출력 길이 제한 해제)
review_vector_search('delicious fruit')

,Content,cos_sim
0,"Title: Delicious!; Content: For anyone who says ""I don't like fruitcake"" or anyone who's never had fruitcake and wonders what all the fuss is about, try this. (As long as you're not allergic to tree nuts or any other ingredient.) It's chock-a-block with nuts and moist fruit. I will definitely be buying more.",0.649525
1,"Title: Delicious .; Content: These plums are sweet and juicy, and the aroma is like perfume. And it doesn't hurt that they are good for you, too.",0.612282
2,"Title: Delicious!; Content: Wonderful! Deep, rich, pure black raspberry syrup! Absolutely delicious on waffles, cheesecake, ice cream, yogurt, drinks, etc. Thrilled to see that there are at least some berry syrup makers who do not feel the need to ""sour"" the flavor of perfect berry products with citric acid!",0.599868
3,"Title: These are Delicious!; Content: Great taste, right price, fabulous snack! It is the only fruit I can get my little one to eat and I can't keep my high school son out of them either. They are great pick-me-ups on the way to my daughter's soccer practice or before my early morning run. And best of all, Amazon ships these right to my door every month. No more finding the right store who carries them. I just set up the automatic recurring shipment once and it works like a charm!",0.591969
4,Title: Delicious!; Content: I have ordered these raisins multiple times. They are always great and arrive timely. I can't go back to store bought chocolate covered raisins now! Love this product.,0.502419


cosine_similarity([emb], query_emb)
[emb] = [[0.123123123, ... , (1536개)]] => (1, 1536)
query_emb = (1, 1536)
코사인 유사도 계산시 [[0.649542]] => (1, 1) => [0, 0]은 값만 빼옴

In [15]:
review_vector_search('Best coffee')

,Content,cos_sim
0,"Title: Best coffee ever!; Content: In my opinion this is the best coffee ever! I've been drinking coffee for 50 plus years and this is what I serve to myself and friends. However, I wish I could find this grind in a pound size, so I could make a full pot rather than just a single cup.",0.622141
1,"Title: Great Coffee; Content: I have a coffee maker that grinds my coffee beans. It's hard to find whole bean decafinated coffee. When I find it in the brand that I like, I am excited. Seattle's Best is my favorite.",0.619311
2,Title: Good coffee.; Content: This is the best Donut Shop Blend out there. Though most k-cups taste a little freeze dried. Keurig needs to work on that. I prefer San Francisco Bay Coffee Company.,0.615143
3,"Title: Better than you-know-who's coffee...; Content: So my wife is a latte freak, and nursing, so decaf is the approved type. After the Senseo left the market, I struggled and found the <a href=""http://www.amazon.com/gp/product/B0047BIWSK"">Aerobie AeroPress Coffee and Espresso Maker</a> which is like a French Press for the 21st century. After getting our recipe figured out, my wife, who's been buying Venti Decaf Latte's at $4 a pop almost daily for years now declares that Seattle's best Level 3 Decaf in her home-made Latte is the best coffee she can get. We've tried other bands, and this is her favorite, hands down!",0.613122
4,"Title: BEST cup of coffee I've ever had!; Content: I thought I'd splurge and try this coffee. It costs much more than other decaf K-Cup options. But I hoped that meant it was better coffee. It IS better coffee. I've never had a better cup of coffee than this. It is excellent when compared to any other decaf or regular coffee I've tried.<br /><br />If you like BOLD, FLAVORFUL decaf coffee try this coffee and you'll really like it.",0.611989
